# NBA 3.1 Feature Selection (LightGBM + SHAP)

Ranks every feature of the combined physician data (`NBA_3_1_Combine_Physician_Data`) against the Kerendia NBRx target (`Target_Variable/HCP_Target_Variable`), joined on `HCP_COHORT_ID`. Adapted from the OE notebook `Feature Selection/3_feature_selection_lgbm_v2_shap_all`.

**Run after:** `HCP_Target_Variable` and `NBA_3_1_Combine_Physician_Data` (same `RUN_NAME`; with `TEST_HCP_SAMPLE` = this notebook's `HCP_SAMPLE`). Cell 3 stops the run if either input is missing or does not hold 28 cohorts for every HCP.

| Step | What it does |
| --- | --- |
| Split | Train = cohorts c1-c24, test = c25-c26; c27-c28 are not used (backtesting holdout) |
| Model | LightGBM, `poisson` objective by default (`TARGET_NBRX` is a non-negative count, mostly 0) |
| Ranking | Gain importance and mean absolute SHAP on all training rows; top `TOP_K` features by SHAP is the selection |
| Outputs | `{VOL31}/hcp_level_model/dev_pipeline/notebook_output/<RUN_NAME>/feature_selection/[_test_sample_<n>/]<OBJECTIVE>/` (3.1 volume only) |

Start with `HCP_SAMPLE = 1000`. The full table (about 7M training rows x 17K features) does not fit in memory with this approach; scaling up needs an HCP sample or a feature pre-filter.

In [0]:
%pip install -q lightgbm shap
dbutils.library.restartPython()

In [0]:
import os, gc, json, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import lightgbm as lgb
from pyspark.sql import functions as F
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

dbutils.widgets.text("RUN_NAME", "combined_2026_11_24")
dbutils.widgets.text("HCP_SAMPLE", "1000")   # = TEST_HCP_SAMPLE of the Combine run; 0 = full combined table
dbutils.widgets.dropdown("OBJECTIVE", "poisson", ["poisson", "tweedie", "regression_l2"])
dbutils.widgets.text("TOP_K", "1000")

RUN_NAME   = dbutils.widgets.get("RUN_NAME").strip()
HCP_SAMPLE = int(dbutils.widgets.get("HCP_SAMPLE") or 0)
OBJECTIVE  = dbutils.widgets.get("OBJECTIVE")
TOP_K      = int(dbutils.widgets.get("TOP_K"))
assert RUN_NAME, "RUN_NAME is required"

# ---- Inputs (outputs of the Combine and Target notebooks) ----
VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
SAMPLE_DIR    = f"_test_sample_{HCP_SAMPLE}/" if HCP_SAMPLE else ""
COMBINED_PATH = f"{DEV}{RUN_NAME}/hcp_combined/{SAMPLE_DIR}kerendia_nba_3_1_hcp_selected_features_data"
DICTIONARY_PATH = f"{DEV}{RUN_NAME}/hcp_combined/{SAMPLE_DIR}kerendia_nba_3_1_hcp_feature_dictionary"
TARGET_PATH   = VOL31 + "notebook_output/target_variable/target_variable"

# ---- Outputs (3.1 volume only) ----
OUTPUT_DIR      = f"{DEV}{RUN_NAME}/feature_selection/{SAMPLE_DIR}{OBJECTIVE}/"
MODEL_DATA_PATH = OUTPUT_DIR + "model_data_parquet"
assert OUTPUT_DIR.startswith(VOL31) and "kerendia_nba_3_volume" not in OUTPUT_DIR, f"refusing to write outside the 3.1 volume: {OUTPUT_DIR}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def out(name):
    return os.path.join(OUTPUT_DIR, name)

# ---- Columns and split ----
TARGET_COL    = "TARGET_NBRX"
N_COHORTS     = 28
TRAIN_COHORTS = (1, 24)    # c1-c24
TEST_COHORTS  = (25, 26)   # c25-c26; c27-c28 = backtesting holdout, not used here
ID_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT_SET", "DATASET", "PREDICTION_MONTH", "LOOKBACK_END_DATE"]

# Source tables left out of the selection (matched on the end of SOURCE_TABLE in the feature dictionary)
# Empty = every Combine feature is a candidate (team decision 2026-11-24: DTC kept in)
# Example: ["kerendia_nba_3_1_hcp_model_dtc_features.parquet"]
EXCLUDE_SOURCE_TABLES = []

# ---- LightGBM (same settings as the OE notebook) ----
SEED = 42
NUM_ITERATIONS, LEARNING_RATE = 250, 0.05
NUM_LEAVES, MIN_DATA_IN_LEAF  = 31, 100
MAX_BIN, BIN_CONSTRUCT_SAMPLE_CNT = 127, 200000
TWEEDIE_VARIANCE_POWER = 1.5
NUM_THREADS = min(os.cpu_count() or 8, 64)
SHAP_BATCH_SIZE = 512
BEESWARM_ROWS   = 1000

print(f"RUN_NAME={RUN_NAME}  HCP_SAMPLE={HCP_SAMPLE}  OBJECTIVE={OBJECTIVE}  TOP_K={TOP_K}  threads={NUM_THREADS}")
print("Combined:", COMBINED_PATH)
print("Target:  ", TARGET_PATH)
print("Output:  ", OUTPUT_DIR)

In [0]:
# Stops here if the Combine or Target notebook has not run (or not for every cohort set)
for p in (COMBINED_PATH, TARGET_PATH):
    assert os.path.exists(p), f"input missing, run its notebook first: {p}"

EXPECTED_COHORTS = {f"c{i}" for i in range(1, N_COHORTS + 1)}

def check_cohorts(df, name, cohort_col):
    a = df.agg(F.count("*").alias("rows"), F.countDistinct("HCP_COHORT_ID").alias("keys"),
               F.countDistinct("PROVIDER_ID").alias("hcps")).first()
    cohorts = {r[0] for r in df.select(cohort_col).distinct().collect()}
    per_hcp = df.groupBy("PROVIDER_ID").agg(F.count("*").alias("n"), F.countDistinct(cohort_col).alias("k"))
    bad_hcps = per_hcp.filter((F.col("n") != N_COHORTS) | (F.col("k") != N_COHORTS)).count()
    print(f"{name:9s} rows {a.rows:,}  keys {a.keys:,}  HCPs {a.hcps:,}  cohorts {len(cohorts)}  HCPs without exactly 28 cohorts: {bad_hcps:,}")
    assert cohorts == EXPECTED_COHORTS, f"{name}: cohorts missing {sorted(EXPECTED_COHORTS - cohorts, key=lambda c: int(c[1:]))}, extra {sorted(cohorts - EXPECTED_COHORTS)}"
    assert a.keys == a.rows, f"{name}: duplicate HCP_COHORT_ID"
    assert bad_hcps == 0 and a.rows == a.hcps * N_COHORTS, f"{name}: not every HCP has 28 cohorts"
    return a

combined = spark.read.format("delta").load(COMBINED_PATH)
target   = spark.read.format("delta").load(TARGET_PATH)
assert TARGET_COL not in combined.columns, f"{TARGET_COL} already in the combined table"

c_stats = check_cohorts(combined, "combined", "COHORT_ID")
t_stats = check_cohorts(target, "target", "cohort_id")
if HCP_SAMPLE:
    assert c_stats.hcps == HCP_SAMPLE, f"combined has {c_stats.hcps} HCPs, expected HCP_SAMPLE={HCP_SAMPLE}"
print(f"Combined columns: {len(combined.columns):,}")

In [0]:
tgt = target.select("HCP_COHORT_ID", F.col("KERENDIA_NBRX").cast("double").alias(TARGET_COL))
joined = (combined.join(tgt, "HCP_COHORT_ID", "left")
          .withColumn("COHORT_N", F.expr("int(substr(COHORT_ID, 2))")))

# The join must not add or lose rows, and TARGET_NBRX must add up to the target table on the same keys
n_joined = joined.count()
assert n_joined == c_stats.rows, f"join changed the row count: {n_joined:,} vs {c_stats.rows:,}"
expected_sum = tgt.join(combined.select("HCP_COHORT_ID"), "HCP_COHORT_ID", "left_semi").agg(F.sum(TARGET_COL)).first()[0]
joined_sum   = joined.agg(F.sum(TARGET_COL)).first()[0]
assert expected_sum == joined_sum, (expected_sum, joined_sum)

no_target = joined.filter(F.col(TARGET_COL).isNull()).select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID")
n_no_target = no_target.count()
print(f"Joined rows {n_joined:,} | TARGET_NBRX total {joined_sum:,.0f} | rows without a target (dropped): {n_no_target:,} "
      f"({no_target.select('PROVIDER_ID').distinct().count():,} HCPs)")
if n_no_target:
    no_target.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(out("rows_without_target"))

model_df = (joined.filter(F.col(TARGET_COL).isNotNull())
            .withColumn("SPLIT", F.when(F.col("COHORT_N").between(*TRAIN_COHORTS), "train")
                                  .when(F.col("COHORT_N").between(*TEST_COHORTS), "test"))
            .filter(F.col("SPLIT").isNotNull()))

split_summary = (model_df.groupBy("SPLIT")
                 .agg(F.count("*").alias("rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
                      F.countDistinct("COHORT_ID").alias("cohorts"), F.sum(TARGET_COL).alias("total_nbrx"),
                      F.avg((F.col(TARGET_COL) > 0).cast("double")).alias("share_rows_with_nbrx"))
                 .orderBy(F.desc("SPLIT")).toPandas())
display(split_summary)

# Plain parquet (overwrite replaces all files), read below with pyarrow as in the OE notebook
model_df.write.mode("overwrite").parquet(MODEL_DATA_PATH)
print("Model data:", MODEL_DATA_PATH)

In [0]:
dataset = ds.dataset(MODEL_DATA_PATH, format="parquet")
schema = dataset.schema

def is_numeric(t):
    return pa.types.is_integer(t) or pa.types.is_floating(t) or pa.types.is_decimal(t) or pa.types.is_boolean(t)

EXCLUDE_COLS = set(ID_COLS) | {TARGET_COL, "COHORT_N", "SPLIT"}
candidate_cols = [f.name for f in schema if f.name not in EXCLUDE_COLS and is_numeric(f.type)]
non_numeric    = [f.name for f in schema if f.name not in EXCLUDE_COLS and not is_numeric(f.type)]
assert candidate_cols, "no numeric feature columns"
print(f"Columns in model data: {len(schema.names):,} | numeric features: {len(candidate_cols):,} | non-numeric skipped: {non_numeric}")

# Leave out features from EXCLUDE_SOURCE_TABLES (see cell 2), using the Combine feature dictionary
feature_dict = spark.read.format("delta").load(DICTIONARY_PATH).select("FEATURE_NAME", "SOURCE_TABLE").toPandas()
src_of = dict(zip(feature_dict.FEATURE_NAME, feature_dict.SOURCE_TABLE.fillna("")))
excluded_df = pd.DataFrame([(c, src_of.get(c, "")) for c in candidate_cols
                            if any(src_of.get(c, "").endswith(t) for t in EXCLUDE_SOURCE_TABLES)],
                           columns=["feature", "source_table"])
for t in EXCLUDE_SOURCE_TABLES:
    assert any(s.endswith(t) for s in src_of.values()), f"EXCLUDE_SOURCE_TABLES entry not in the feature dictionary: {t}"
excluded_df.to_csv(out("excluded_source_features.csv"), index=False)
candidate_cols = [c for c in candidate_cols if c not in set(excluded_df.feature)]
print(f"Left out by EXCLUDE_SOURCE_TABLES: {len(excluded_df):,} features | candidates now {len(candidate_cols):,}")

In [0]:
def load_split(split):
    """float32 X (rows x features, NaN = missing) and y for one split, filled batch by batch into preallocated arrays."""
    flt = ds.field("SPLIT") == split
    n = dataset.count_rows(filter=flt)
    X = np.empty((n, len(candidate_cols)), dtype=np.float32)
    y = np.empty(n, dtype=np.float32)
    pos = 0
    for b in dataset.to_batches(columns=[TARGET_COL] + candidate_cols, filter=flt, batch_size=8192):
        k = b.num_rows
        y[pos:pos + k] = b.column(0).cast(pa.float32()).to_numpy(zero_copy_only=False)
        for j in range(len(candidate_cols)):
            X[pos:pos + k, j] = b.column(j + 1).cast(pa.float32(), safe=False).to_numpy(zero_copy_only=False)
        pos += k
    assert pos == n, (split, pos, n)
    X[np.isinf(X)] = np.nan
    assert np.isfinite(y).all(), f"{split}: non-finite target"
    return X, y

t0 = time.time()
X_train_all, y_train = load_split("train")
X_test_all, y_test   = load_split("test")
print(f"Loaded in {time.time() - t0:,.1f}s | X_train {X_train_all.shape} | X_test {X_test_all.shape} | "
      f"{(X_train_all.nbytes + X_test_all.nbytes) / 1024**3:,.2f} GiB")
assert len(y_train) == int(split_summary.loc[split_summary.SPLIT == "train", "rows"].iloc[0])
assert len(y_test)  == int(split_summary.loc[split_summary.SPLIT == "test", "rows"].iloc[0])

In [0]:
# A feature with one value (or all missing) on the training rows cannot be split on, so it gets no importance
with np.errstate(all="ignore"):
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        col_min = np.nanmin(X_train_all, axis=0)
        col_max = np.nanmax(X_train_all, axis=0)
constant = ~(col_max > col_min)          # also True when the column is all NaN
keep_idx = np.where(~constant)[0]

constant_df = pd.DataFrame({"feature": np.array(candidate_cols)[constant], "value": col_min[constant]})
constant_df.to_csv(out("constant_features_dropped.csv"), index=False)

feature_cols = [candidate_cols[i] for i in keep_idx]
n_features = len(feature_cols)
X_train = np.ascontiguousarray(X_train_all[:, keep_idx]); X_test = np.ascontiguousarray(X_test_all[:, keep_idx])
del X_train_all, X_test_all; gc.collect()

feature_aliases = [f"f_{i}" for i in range(n_features)]   # LightGBM-safe names; mapped back below
feature_mapping_df = pd.DataFrame({"feature_index": np.arange(n_features), "feature_alias": feature_aliases, "feature": feature_cols})
feature_mapping_df.to_csv(out("feature_alias_mapping.csv"), index=False)
print(f"Candidate features {len(candidate_cols):,} | constant on train (dropped) {int(constant.sum()):,} | used {n_features:,}")

In [0]:
target_summary = pd.DataFrame([{
    "split": name, "rows": len(y), "rows_with_nbrx": int((y > 0).sum()), "share_with_nbrx": float((y > 0).mean()),
    "total_nbrx": float(y.sum()), "mean": float(y.mean()), "max": float(y.max())}
    for name, y in [("train", y_train), ("test", y_test)]])
display(target_summary)
assert target_summary.loc[0, "rows_with_nbrx"] > 0, "no NBRx in the training rows"

In [0]:
dataset_params = {"max_bin": MAX_BIN, "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT, "feature_pre_filter": False}
train_data = lgb.Dataset(X_train, label=y_train, feature_name=feature_aliases, free_raw_data=False, params=dataset_params)
test_data  = lgb.Dataset(X_test, label=y_test, feature_name=feature_aliases, reference=train_data, free_raw_data=False, params=dataset_params)

lgb_params = {
    "objective": OBJECTIVE,
    "metric": ["rmse", "l1"] + ([OBJECTIVE] if OBJECTIVE != "regression_l2" else []),
    "boosting_type": "gbdt",
    "learning_rate": LEARNING_RATE, "num_leaves": NUM_LEAVES, "min_data_in_leaf": MIN_DATA_IN_LEAF,
    "feature_fraction": 1.0, "bagging_fraction": 1.0, "bagging_freq": 0,
    "max_bin": MAX_BIN, "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "force_col_wise": True, "num_threads": NUM_THREADS, "seed": SEED, "verbosity": -1,
}
if OBJECTIVE == "tweedie":
    lgb_params["tweedie_variance_power"] = TWEEDIE_VARIANCE_POWER

t0 = time.time()
eval_log = {}
booster = lgb.train(lgb_params, train_data, num_boost_round=NUM_ITERATIONS,
                    valid_sets=[train_data, test_data], valid_names=["train", "test"],
                    callbacks=[lgb.log_evaluation(period=25), lgb.record_evaluation(eval_log)])
print(f"Training done in {time.time() - t0:,.1f}s")
# LightGBM appends to the model file after writing it; UC volumes do not support append, so save locally and copy
import shutil, tempfile
_local_model = os.path.join(tempfile.mkdtemp(), "lightgbm_model.txt")
booster.save_model(_local_model)
shutil.copyfile(_local_model, out("lightgbm_model.txt"))
assert os.path.getsize(out("lightgbm_model.txt")) == os.path.getsize(_local_model), "model copy incomplete"
print("Model:", out("lightgbm_model.txt"))

In [0]:
pred_train = booster.predict(X_train, num_threads=NUM_THREADS)
pred_test  = booster.predict(X_test, num_threads=NUM_THREADS)

# MAPE is left out: most rows have 0 NBRx. Total predicted vs actual NBRx shows the overall calibration.
def metrics(y, p, name):
    return {"split": name, "rows": len(y), "r2": r2_score(y, p), "rmse": float(np.sqrt(mean_squared_error(y, p))),
            "mae": mean_absolute_error(y, p), "actual_total_nbrx": float(y.sum()), "predicted_total_nbrx": float(p.sum()),
            "predicted_over_actual": float(p.sum() / y.sum()) if y.sum() else np.nan}

metrics_df = pd.DataFrame([metrics(y_train, pred_train, "train (c1-c24)"), metrics(y_test, pred_test, "test (c25-c26)")])
metrics_df.to_csv(out("train_test_metrics.csv"), index=False)
display(metrics_df)

In [0]:
importance_df = pd.DataFrame({
    "feature_index": np.arange(n_features), "feature_alias": feature_aliases, "feature": feature_cols,
    "gain_importance": booster.feature_importance(importance_type="gain").astype(float),
    "split_importance": booster.feature_importance(importance_type="split").astype(float),
}).sort_values("gain_importance", ascending=False).reset_index(drop=True)
importance_df["gain_rank"] = np.arange(1, n_features + 1)
importance_df.to_csv(out("all_feature_importance_gain.csv"), index=False)
print(f"Features used by the model (gain > 0): {(importance_df.gain_importance > 0).sum():,} of {n_features:,}")
display(importance_df.head(30))

In [0]:
# pred_contrib returns one SHAP value per feature + a bias column, in the model's raw score
# (log scale for poisson / tweedie, so the ranking is on log expected NBRx)
shap_sum = np.zeros(n_features, dtype=np.float64)
shap_abs_sum = np.zeros(n_features, dtype=np.float64)
t0 = time.time()
for start in range(0, X_train.shape[0], SHAP_BATCH_SIZE):
    contrib = booster.predict(X_train[start:start + SHAP_BATCH_SIZE], pred_contrib=True, num_threads=NUM_THREADS)[:, :n_features]
    shap_sum += contrib.sum(axis=0)
    shap_abs_sum += np.abs(contrib).sum(axis=0)
    del contrib
n_shap_rows = X_train.shape[0]
print(f"SHAP on {n_shap_rows:,} rows x {n_features:,} features in {time.time() - t0:,.1f}s")

mean_abs = shap_abs_sum / n_shap_rows
shap_rank_df = pd.DataFrame({
    "feature_index": np.arange(n_features), "feature_alias": feature_aliases, "feature": feature_cols,
    "mean_shap_train": shap_sum / n_shap_rows, "mean_abs_shap_train": mean_abs,
    "percent_impact_shap": 100.0 * mean_abs / mean_abs.sum() if mean_abs.sum() > 0 else 0.0,
}).sort_values("mean_abs_shap_train", ascending=False).reset_index(drop=True)
shap_rank_df.insert(0, "shap_rank", np.arange(1, n_features + 1))
shap_rank_df["cumulative_percent_impact_shap"] = shap_rank_df["percent_impact_shap"].cumsum()
shap_rank_df = shap_rank_df.merge(importance_df[["feature", "gain_rank", "gain_importance"]], on="feature", how="left")
shap_rank_df.to_csv(out("shap_ranking_all_features.csv"), index=False)
print(f"Features with SHAP > 0: {(shap_rank_df.mean_abs_shap_train > 0).sum():,}")
display(shap_rank_df.head(30))

In [0]:
# Features the model never used have SHAP = 0; they are not selected even if TOP_K is larger
ranked = shap_rank_df[shap_rank_df.mean_abs_shap_train > 0]
top_100_df = ranked.head(100).reset_index(drop=True)
selected_df = ranked.head(TOP_K).reset_index(drop=True)
top_100_df.to_csv(out("top_100_features_by_shap.csv"), index=False)
selected_df.to_csv(out(f"final_selected_top_{TOP_K}_features_by_shap.csv"), index=False)
print(f"Selected {len(selected_df):,} features (TOP_K={TOP_K}); they carry "
      f"{selected_df.percent_impact_shap.sum():.1f}% of total mean |SHAP|")
display(selected_df.head(50))

In [0]:
import matplotlib.pyplot as plt
import shap

top30 = top_100_df.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top30.feature, top30.percent_impact_shap)
ax.set_xlabel("% of total mean |SHAP|"); ax.set_title(f"Top 30 features by SHAP impact ({OBJECTIVE}, train c1-c24)")
plt.tight_layout(); fig.savefig(out("top_30_shap_impact.png"), dpi=120); plt.show()

rng = np.random.default_rng(SEED)
rows = np.sort(rng.choice(X_train.shape[0], size=min(BEESWARM_ROWS, X_train.shape[0]), replace=False))
idx = top_100_df.head(30).feature_index.to_numpy()
contrib = booster.predict(X_train[rows], pred_contrib=True, num_threads=NUM_THREADS)[:, idx]
plt.figure()
shap.summary_plot(contrib, pd.DataFrame(X_train[rows][:, idx], columns=top_100_df.head(30).feature.tolist()),
                  max_display=30, show=False, plot_size=(14, 12))
plt.tight_layout(); plt.savefig(out("top_30_shap_beeswarm.png"), dpi=120); plt.show()

In [0]:
summary = {
    "run_name": RUN_NAME, "hcp_sample": HCP_SAMPLE, "objective": OBJECTIVE, "top_k": TOP_K,
    "combined_path": COMBINED_PATH, "target_path": TARGET_PATH, "output_dir": OUTPUT_DIR,
    "combined_hcps": int(c_stats.hcps), "combined_rows": int(c_stats.rows), "rows_without_target": int(n_no_target),
    "train_rows": int(len(y_train)), "test_rows": int(len(y_test)),
    "candidate_features": len(candidate_cols), "constant_dropped": int(constant.sum()), "features_used": n_features,
    "features_with_shap": int((shap_rank_df.mean_abs_shap_train > 0).sum()), "selected_features": len(selected_df),
    "metrics": metrics_df.to_dict(orient="records"), "lgb_params": lgb_params, "num_iterations": NUM_ITERATIONS,
    "written_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(out("run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps({k: v for k, v in summary.items() if k not in ("metrics", "lgb_params")}, indent=2))
print("Files:", sorted(os.listdir(OUTPUT_DIR)))